Lanette Tyler
ST554-601
Spring 2026
Project 2, Parts 1b and 2

# Project 2: Part 1b

## Introduction   
The purpose of Part 1 of the project is to create a data quality class in pyspark. The python class, called SparkDataCheck, wraps a Spark, SQL-style data frame and provides functionality for cleaning and validating the data frame. The class can take in a Spark, SQL-style data frame through the .\_\_init\_\_ function, a csv file throught the .from_csv class method, or a standard pandas data frame through the .from_pandas_df class method. The class provides validation and summarization methods for numeric and string columns and documents NULL values in columns. The code for SparkDataCheck class is contained in the python script project2_part1.py.

The purpose of Part 2 of the project is to test and demonstrate SparkDataCheck class on the [provided data](https://www4.stat.ncsu.edu/online/datasets/air.csv), which is documented below. The dataset is copied into the working directory.

## Preliminary Data Steps

In [5]:
#import modules
from pyspark.sql import SparkSession
import pandas as pd
import pyspark.sql.functions as F

In [6]:
#create spark session
spark = SparkSession.builder.appName("my_app") \
    .config("pyspark.sql.ansi.enabled", "false").getOrCreate()

In [7]:
#import SparkDataCheck class (project2_part1.py)
import project2_part1

In [58]:
#prep data (starting with air.csv, ending up with updated air_mod.csv and pandas data frame air_pdf)

#read in data, check for -200 values (which represent missing values in this data set)
air_pdf = pd.read_csv("air.csv")
print(air_pdf["T"].min())
print(air_pdf["RH"].min())
print(air_pdf["AH"].min())

#replace -200 values with None, check min values again (no longer -200)
air_pdf = air_pdf.replace(-200, None)
print(air_pdf["T"].min())
print(air_pdf["RH"].min())
print(air_pdf["AH"].min())

#delete first column with problematic problem name (column seems to be an index)
air_pdf = air_pdf.iloc[:, 1:]

#add string columns for demonstrating string column validation method
for i in range(len(air_pdf)):
        if air_pdf.loc[i, "RH"] == None:
               air_pdf.loc[i, "RH_cat"] = None
        elif air_pdf.loc[i, "RH"] >= 50:
               air_pdf.loc[i, "RH_cat"] = "High"
        else:
               air_pdf.loc[i, "RH_cat"] = "Low"
                
for i in range(len(air_pdf)):
        if air_pdf.loc[i, "AH"] == None:
               air_pdf.loc[i, "AH_cat"] = None
        elif air_pdf.loc[i, "AH"] >= 1.03:
               air_pdf.loc[i, "AH_cat"] = "High"
        else:
               air_pdf.loc[i, "AH_cat"] = "Low"

#clean up column names
air_pdf.columns = air_pdf.columns.str.replace(".", "_")
                
#export to air_mod.csv
air_pdf.to_csv("air_mod.csv", index = False)

#take a look at modified data frame
air_pdf.head()

-200.0
-200.0
-200.0
-1.9
9.2
0.1847


,Date,Time,CO(GT),PT08_S1(CO),NMHC(GT),C6H6(GT),PT08_S2(NMHC),NOx(GT),PT08_S3(NOx),NO2(GT),PT08_S4(NO2),PT08_S5(O3),T,RH,AH,RH_cat,AH_cat
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578,Low,Low
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255,Low,Low
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502,High,Low
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867,High,Low
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888,High,Low


## Create instance of SparkDataCheck Class from air.csv file and demonstrate methods on it

### Create and view instance

In [61]:
#create an instance
sdc_air = project2_part1.SparkDataCheck.from_csv(spark, "air_mod.csv")

#view the instance
print(sdc_air)
print("\n")
print(sdc_air.df)
print("\n")
sdc_air.df.show(3)



DataFrame[Date: string, Time: timestamp, CO(GT): double, PT08_S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08_S2(NMHC): int, NOx(GT): int, PT08_S3(NOx): int, NO2(GT): int, PT08_S4(NO2): int, PT08_S5(O3): int, T: double, RH: double, AH: double, RH_cat: string, AH_cat: string]


+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|
|3/10/2004|2026-03-21 19:00:00|   2.0|       

### Demonstrate .numColVal() method for validating numeric columns   
Check values in a user-selected numeric column to see if they fall within inclusive, user-defined limits. A resulting column of boolean values is appended to the SparkDataCheck object's data frame attribute.

In [62]:
#demonstrate .numColVal method, invalid column name
sdc_air.numColVal("temp", upper_bound = 20, lower_bound = 15)

Please enter a valid column name as a string.


In [63]:
#demonstrate .numColVal method, non-numeric column
sdc_air.numColVal("RH_cat", upper_bound  = 75, lower_bound = 25)

Please enter a column with numeric data.


DataFrame[Date: string, Time: timestamp, CO(GT): double, PT08_S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08_S2(NMHC): int, NOx(GT): int, PT08_S3(NOx): int, NO2(GT): int, PT08_S4(NO2): int, PT08_S5(O3): int, T: double, RH: double, AH: double, RH_cat: string, AH_cat: string]

In [64]:
#demontrate .numColVal method, no upper or lower bounds entered
sdc_air.numColVal("T")

Please enter a lower and/or upper bound.


DataFrame[Date: string, Time: timestamp, CO(GT): double, PT08_S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08_S2(NMHC): int, NOx(GT): int, PT08_S3(NOx): int, NO2(GT): int, PT08_S4(NO2): int, PT08_S5(O3): int, T: double, RH: double, AH: double, RH_cat: string, AH_cat: string]

In [65]:
#demonstrate .numColVal method, both upper and lower bounds provided
sdc_air.numColVal("AH", lower_bound = 0.25, upper_bound = 0.75).show(3)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|
|3/10/2004|2026-03-21 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|   Low|   Low|                           true|


In [66]:
#demonstrate .numColVal, only upper bound provided
sdc_air.numColVal("RH", upper_bound = 50).show(3)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|RH <= 50|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|    true|
|3/10/2004|2026-03-21 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|   Low|   L

In [67]:
#demonstrate .numColVal, only lower bound provided
sdc_air.numColVal("T", lower_bound = 0).show(3)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|RH <= 50|T >= 0|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|    true|  true|
|3/10/2004|2026-03-21 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|

### Demonstrate .strColVal() method for validating string columns
Check values in a user-selected string column to see if they fall within a user-defined list of string values. A column of boolean values is appended to the sparkDataCheck object's data frame attribute.

In [68]:
#demonstrate .strColVal, invalid column name
sdc_air.strColVal("time", levels = ["High", "Low"])

Please enter a valid column name as a string.


In [69]:
#demonstrate .strColVal, column wrong format (not string)
sdc_air.strColVal("T", levels = ["High", "Low"])

Please enter a column with string data.


DataFrame[Date: string, Time: timestamp, CO(GT): double, PT08_S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08_S2(NMHC): int, NOx(GT): int, PT08_S3(NOx): int, NO2(GT): int, PT08_S4(NO2): int, PT08_S5(O3): int, T: double, RH: double, AH: double, RH_cat: string, AH_cat: string, ((AH >= 0.25) AND (AH <= 0.75)): boolean, RH <= 50: string, T >= 0: string]

In [70]:
#demonstrate .strColVal method, check for user entered values when all column values are in user entered values
sdc_air.strColVal("RH_cat", levels = ["High", "Low"])
sdc_air.df.show

<bound method DataFrame.show of DataFrame[Date: string, Time: timestamp, CO(GT): double, PT08_S1(CO): int, NMHC(GT): int, C6H6(GT): double, PT08_S2(NMHC): int, NOx(GT): int, PT08_S3(NOx): int, NO2(GT): int, PT08_S4(NO2): int, PT08_S5(O3): int, T: double, RH: double, AH: double, RH_cat: string, AH_cat: string, ((AH >= 0.25) AND (AH <= 0.75)): boolean, RH <= 50: string, T >= 0: string, RH_catCheck: boolean]>

In [71]:
#RH_catCheck column has been added, values are limited to user-entered values
sdc_air.df.show(3)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|RH <= 50|T >= 0|RH_catCheck|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|    true|  true|       true|
|3/10/2004|2026-03-21 19:00:00|   2.0|       1292|     112|     9.4|          955|    10

In [72]:
#demonstrate .strColVal, check for user entered values when all column values are not user-entered values
sdc_air.strColVal("RH_cat", levels = ["high", "low"])
sdc_air.df.show(3)
#since it's the same column validated as in previous step, new column name is the same and previous RH_catCheck column was replaced

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|RH <= 50|T >= 0|RH_catCheck|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|    true|  true|      false|
|3/10/2004|2026-03-21 19:00:00|   2.0|       1292|     112|     9.4|          955|    10

### Demonstrate .nullCheck() method
Check values in a user-selected column to see if they are NULL. A column of boolean values is appeneded to the sparkDataCheck object's data frame attribute.

In [73]:
sdc_air.nullCheck("Time")
sdc_air.df.show(3)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+----------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|RH <= 50|T >= 0|RH_catCheck|TimeIsNull|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+----------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|    true|  true|      false|     false|
|3/10/2004|2026-03-21 19:00:00|   2.0|      

In [74]:
sdc_air.nullCheck("T")
sdc_air.df.show(3)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+----------+-------+
|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|((AH >= 0.25) AND (AH <= 0.75))|RH <= 50|T >= 0|RH_catCheck|TimeIsNull|TIsNull|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+-------------------------------+--------+------+-----------+----------+-------+
|3/10/2004|2026-03-21 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|                          false|    true|  true|      false|     false|  false|
|3/10/2004|2

### Demonstrate .numColSummarizer() method for summarizing numeric data columns

In [75]:
#invalid column name
sdc_air.numColSummarizer(column = "temp")

This is not a valid column name.


In [76]:
#non-numeric column
sdc_air.numColSummarizer(column = "RH_cat")

Please enter a column with numeric data.


In [77]:
#one column input, no grouping variable
sdc_air.numColSummarizer(column = "NMHC(GT)")

,min(NMHC(GT)),max(NMHC(GT))
0,7,1189


In [78]:
#one column input, with grouping variable
sdc_air.numColSummarizer(column = "NMHC(GT)", gr_var = "RH_cat")

,RH_cat,min(NMHC(GT)),max(NMHC(GT))
0,High,7,1129
1,Low,9,1189
2,None,66,512


In [79]:
#no column input, without grouping variable
sdc_air.numColSummarizer()

,summary,CO(GT),PT08_S1(CO),NMHC(GT),C6H6(GT),PT08_S2(NMHC),NOx(GT),PT08_S3(NOx),NO2(GT),PT08_S4(NO2),PT08_S5(O3),T,RH,AH
0,min,0.1,647,7,0.1,383,2,322,2,551,221,-1.9,9.2,0.1847
1,max,11.9,2040,1189,63.7,2214,1479,2683,340,2775,2523,44.6,88.7,2.231


In [80]:
#no column input, with grouping variable
sdc_air.numColSummarizer(gr_var = "RH_cat")

,RH_cat,min(CO(GT)),min(PT08_S1(CO)),min(NMHC(GT)),min(C6H6(GT)),min(PT08_S2(NMHC)),min(NOx(GT)),min(PT08_S3(NOx)),min(NO2(GT)),min(PT08_S4(NO2)),...,max(C6H6(GT)),max(PT08_S2(NMHC)),max(NOx(GT)),max(PT08_S3(NOx)),max(NO2(GT)),max(PT08_S4(NO2)),max(PT08_S5(O3)),max(T),max(RH),max(AH)
0,High,0.1,678.0,7,0.1,383.0,2,322.0,2,669.0,...,63.7,2214.0,1479,2095.0,322,2775.0,2523.0,29.8,88.7,2.2310
1,Low,0.1,647.0,9,0.2,387.0,4,330.0,8,551.0,...,44.7,1866.0,1290,2683.0,333,2691.0,2475.0,44.6,49.9,2.1144
2,None,0.3,NaN,66,NaN,NaN,21,NaN,35,NaN,...,NaN,NaN,1310,NaN,340,NaN,NaN,NaN,NaN,NaN


### Demonstrate .strColSummarizer() method for summarizing string data columns

In [81]:
#invalid column name
sdc_air.strColSummarizer(columns = ["rh_cat"])

Please enter a valid column name.


In [82]:
#not string data type
sdc_air.strColSummarizer(columns = ["Time"])

This column is not string type data. Please enter a column of string data type.


In [84]:
#enter one column
sdc_air.strColSummarizer(columns = ["RH_cat"])

,RH_cat,count
0,High,4408
1,Low,4583
2,None,366


In [85]:
#two columns entered, but only one string data type column
sdc_air.strColSummarizer(columns = ["RH_cat", "T"])

,RH_cat,count
0,High,4408
1,Low,4583
2,None,366


In [87]:
#two string columns entered
sdc_air.strColSummarizer(columns = ["RH_cat", "AH_cat"])

,RH_cat,AH_cat,count
0,None,None,366
1,High,Low,2242
2,Low,Low,2581
3,Low,High,2002
4,High,High,2166


## Create an instance of SparkDataCheck Class from pandas data frame and demonstrate one method call

In [94]:
#create an instance from pandas data frame
sdc_air2 = project2_part1.SparkDataCheck.from_pandas_df(spark, air_pdf)
sdc_air2.df.show(3)

+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+
|     Date|    Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|RH_cat|AH_cat|
+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+------+
|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|   Low|   Low|
|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|   Low|   Low|
|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|  High|   Low|
+---------+--------+------+-----------+--------+--------+-

In [97]:
#demonstrate a method call
sdc_air2.numColSummarizer(column = "PT08_S5(O3)")

,min(PT08_S5(O3)),max(PT08_S5(O3))
0,221,2523


# Project 2: Part 2

##